# Notebook 8.2 - Deployment

Create or update `mnist-serving-endpoint`. The first deployment sends `100%` traffic to the `champion` alias. The second deployment cell updates the endpoint to `50/50` between `champion` and `challenger`.

In [ ]:
import time
import mlflow

from databricks.sdk import WorkspaceClient
from mlflow import MlflowClient
from mlflow.deployments import get_deploy_client

dbutils.widgets.text("catalog_name", "ai_ml_in_practice")
dbutils.widgets.text("schema_name", "mlops_workshop")
dbutils.widgets.text("registered_model_name", "mnist_cnn")
dbutils.widgets.text("endpoint_name", "mnist-serving-endpoint")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
registered_model_name = dbutils.widgets.get("registered_model_name")
endpoint_name = dbutils.widgets.get("endpoint_name")

full_model_name = f"{catalog_name}.{schema_name}.{registered_model_name}"

mlflow.set_registry_uri("databricks-uc")
deploy_client = get_deploy_client("databricks")
mlflow_client = MlflowClient()
workspace_client = WorkspaceClient()


def get_version_for_alias(model_name, alias_name):
    model_version = mlflow_client.get_model_version_by_alias(name=model_name, alias=alias_name)
    return str(model_version.version)


def endpoint_exists(name):
    try:
        workspace_client.serving_endpoints.get(name)
        return True
    except Exception:
        return False


def create_or_update_endpoint(name, endpoint_config):
    if endpoint_exists(name):
        try:
            return deploy_client.update_endpoint(config=endpoint_config)
        except TypeError:
            return "Updating endpoint failed"
    try:
        return deploy_client.create_endpoint(config=endpoint_config)
    except TypeError:
        return "Creating endpoint failed"


def wait_until_ready(name, timeout_seconds=1800):
    def normalize_enum(value):
        text = str(value)
        return text.split(".")[-1] if "." in text else text

    start = time.time()
    while time.time() - start < timeout_seconds:
        endpoint = workspace_client.serving_endpoints.get(name)
        state = getattr(endpoint, "state", None)
        ready = getattr(state, "ready", None)
        config_update = getattr(state, "config_update", None)
        print({"ready": str(ready), "config_update": str(config_update)})
        ready_value = normalize_enum(ready).upper()
        config_update_value = normalize_enum(config_update).upper()
        if ready_value == "READY" and config_update_value in {"NOT_UPDATING", ""}:
            return endpoint
        time.sleep(30)
    raise TimeoutError(f"Endpoint {name} did not become ready in time")

In [ ]:
champion_version = get_version_for_alias(full_model_name, "champion")

initial_config = {
    "name": endpoint_name,
    "ai_gateway": {
        "usage_tracking_config": {
            "enabled": True
        }
    },
    "config": {
        "served_entities": [
            {
                "name": "champion",
                "entity_name": full_model_name,
                "entity_version": champion_version,
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            }
        ],
        "traffic_config": {
            "routes": [
                {"served_model_name": "champion", "traffic_percentage": 100}
            ]
        }
    }
}

create_or_update_endpoint(initial_config)

wait_until_ready(endpoint_name)
workspace_client.serving_endpoints.get(endpoint_name)

In [ ]:
champion_version = get_version_for_alias(full_model_name, "champion")
challenger_version = get_version_for_alias(full_model_name, "challenger")

canary_config = {
    "name": endpoint_name,
    "ai_gateway": {
        "usage_tracking_config": {
            "enabled": True
        }
    },
    "config": {
        "served_entities": [
            {
                "name": "champion",
                "entity_name": full_model_name,
                "entity_version": champion_version,
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            },
            {
                "name": "challenger",
                "entity_name": full_model_name,
                "entity_version": challenger_version,
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            },
        ],
        "traffic_config": {
            "routes": [
                {"served_model_name": "champion", "traffic_percentage": 50},
                {"served_model_name": "challenger", "traffic_percentage": 50},
            ]
        }
    }
}

create_or_update_endpoint(canary_config)
wait_until_ready(endpoint_name)
workspace_client.serving_endpoints.get(endpoint_name)